# 11. SegFormer-B3 - river plume segmentation

Fine-tuning of `nvidia/segformer-b3-finetuned-ade-512-512` (HuggingFace
Transformers) for binary river-plume segmentation on the Sentinel-2/Landsat
composites of `split_data`.

Setup used in the study: input 512x512, torchvision-style ImageNet
normalization, AdamW with two parameter groups (decode head lr 1e-4, trainable
encoder parameters lr 1e-5), weight decay 0.01, CosineAnnealingLR over the
epoch budget, gradient clipping 1.0, batch size 8. The first 70% of the
encoder parameter tensors are frozen. Early stopping monitors the validation
loss (patience 8); validation Dice is logged for reference. Training uses the
shared augmentation policy of the study (flips, rotation within 25 degrees,
adaptive photometric transforms - identical to the CNN models).

In [ ]:
import gc
import os
import random
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from transformers import SegformerForSemanticSegmentation

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

## 1. Configuration

In [ ]:
# ============================================================================
# CONFIGURATION - values reproduce the training run used in the paper
# ============================================================================

PRETRAINED = "nvidia/segformer-b3-finetuned-ade-512-512"

TRAIN_DIR = "split_data/train"
VAL_DIR = "split_data/val"
TEST_IMG_DIR = "split_data/test/images"
TEST_MASK_DIR = "split_data/test/Masks"

NUM_CLASSES = 2                 # 0 = background, 1 = river plume
IMG_SIZE = 512
BATCH_SIZE = 8
NUM_WORKERS = 0                 # increase on Linux if desired

EPOCHS = 100                     # epoch budget; also defines the cosine schedule length
LEARNING_RATE = 1e-4            # decode head learning rate
ENCODER_LR_MULT = 0.1           # trainable encoder parameters train at LEARNING_RATE * 0.1
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0                 # gradient norm clipping
FREEZE_RATIO = 0.7              # fraction of encoder parameter tensors to freeze
PATIENCE = 8                    # early stopping on validation loss

SAVE_PATH = "best_segformer_model.pth"

PRED_THRESHOLD = 0.5
PROBABILITY_EXPORT_DIR = "probability_masks/test/segformer"
BINARY_EXPORT_DIR = "test_masks/segformer"


def optimize_memory():
    """Release cached GPU memory between epochs."""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

## 2. Augmentation (shared study-wide policy)

In [ ]:
class Augmentor:
    """Shared augmentation policy of the study.

    Identical for every model except YOLO11-seg (which uses the native
    Ultralytics augmentation). Operates on RGB images in [0, 255] (uint8 or
    float32) and binary masks; returns an RGB float32 image in [0, 255] and a
    2D uint8 mask in {0, 1}. Photometric parameter ranges depend on the mean
    brightness of the image so that dark scenes are not darkened further and
    bright scenes are not over-exposed.
    """

    def __init__(self, rotation_range=(-25, 25), rotation_prob=0.9, flip_prob=0.5,
                 min_mean_allowed=60.0):
        self.rotation_range = rotation_range
        self.rotation_prob = rotation_prob
        self.flip_prob = flip_prob
        self.min_mean_allowed = min_mean_allowed

    def augment(self, img_rgb, mask):
        img = np.clip(img_rgb, 0, 255).astype(np.uint8)
        mask_in = (mask > 0.5).astype(np.uint8)

        h, w = img.shape[:2]

        # 1) Geometric augmentations.
        if np.random.rand() < self.flip_prob:
            img = cv2.flip(img, 1)
            mask_in = cv2.flip(mask_in, 1)
        if np.random.rand() < self.flip_prob:
            img = cv2.flip(img, 0)
            mask_in = cv2.flip(mask_in, 0)

        if np.random.rand() < self.rotation_prob:
            angle = float(np.random.uniform(self.rotation_range[0], self.rotation_range[1]))
            Mrot = cv2.getRotationMatrix2D((w / 2.0, h / 2.0), angle, 1.0)
            img = cv2.warpAffine(img, Mrot, (w, h), flags=cv2.INTER_LINEAR,
                                 borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))
            mask_in = cv2.warpAffine(mask_in, Mrot, (w, h), flags=cv2.INTER_NEAREST,
                                     borderMode=cv2.BORDER_CONSTANT, borderValue=0)

        # 2) Adaptive photometric transforms driven by the mean brightness.
        gray0 = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).astype(np.float32)
        mean0 = float(gray0.mean())

        dark_thresh = 80.0
        bright_thresh = 200.0

        if mean0 < dark_thresh:
            # Dark scenes: brighten gently, never darken.
            gamma = float(np.random.uniform(0.8, 1.05))
            alpha = float(np.random.uniform(0.9, 1.15))
            beta = float(np.random.uniform(0, 40))
            apply_clahe = (np.random.rand() < 0.3)
            apply_hsv = (np.random.rand() < 0.25)
        elif mean0 > bright_thresh:
            # Very bright scenes: allow slight darkening.
            gamma = float(np.random.uniform(0.95, 1.25))
            alpha = float(np.random.uniform(0.85, 1.25))
            beta = float(np.random.uniform(-40, 20))
            apply_clahe = (np.random.rand() < 0.5)
            apply_hsv = (np.random.rand() < 0.5)
        else:
            # Mid-tone scenes: moderate transforms.
            gamma = float(np.random.uniform(0.9, 1.1))
            alpha = float(np.random.uniform(0.92, 1.2))
            beta = float(np.random.uniform(-20, 30))
            apply_clahe = (np.random.rand() < 0.45)
            apply_hsv = (np.random.rand() < 0.4)

        # Gamma correction in [0, 1] float space.
        img_f = np.clip(img.astype(np.float32) / 255.0, 0.0, 1.0)
        if abs(gamma - 1.0) > 1e-6:
            img_f = np.power(img_f, gamma)
        img_gc = np.clip(img_f * 255.0, 0, 255).astype(np.float32)

        # Mean-centred linear contrast plus brightness shift.
        gray = cv2.cvtColor(img_gc.astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float32)
        mean_l = gray.mean()
        img_out = (img_gc - mean_l) * alpha + mean_l + beta

        # Moderate CLAHE on the L channel.
        if apply_clahe:
            lab = cv2.cvtColor(np.clip(img_out, 0, 255).astype(np.uint8), cv2.COLOR_RGB2LAB)
            L, A, B = cv2.split(lab)
            clip_limit = np.random.uniform(1.0, 3.0)
            clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(8, 8))
            L = clahe.apply(L)
            lab = cv2.merge([L, A, B])
            img_out = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB).astype(np.float32)

        # Gentle HSV tweak.
        if apply_hsv:
            hsv = cv2.cvtColor(np.clip(img_out, 0, 255).astype(np.uint8),
                               cv2.COLOR_RGB2HSV).astype(np.float32)
            h_ch, s_ch, v_ch = cv2.split(hsv)
            if mean0 < dark_thresh:
                sat_mul = np.random.uniform(0.9, 1.15)
                val_shift = np.random.uniform(0, 30)
            else:
                sat_mul = np.random.uniform(0.85, 1.25)
                val_shift = np.random.uniform(-30, 30)
            s_ch = np.clip(s_ch * sat_mul, 0, 255)
            v_ch = np.clip(v_ch + val_shift, 0, 255)
            hsv2 = cv2.merge([h_ch, s_ch, v_ch]).astype(np.uint8)
            img_out = cv2.cvtColor(hsv2, cv2.COLOR_HSV2RGB).astype(np.float32)

        # Strict guard against overly dark outputs.
        gray_after = cv2.cvtColor(np.clip(img_out, 0, 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
        mean_after = float(gray_after.mean())
        if mean_after < self.min_mean_allowed:
            img_out = img_out + (self.min_mean_allowed - mean_after)

        img_final = np.clip(img_out, 0, 255).astype(np.float32)
        mask_final = (mask_in > 0).astype(np.uint8)

        return img_final, mask_final

## 3. Preprocessing

In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def preprocess_image(image_f32):
    """Resize to IMG_SIZE and apply torchvision-style ImageNet normalization.

    Input: RGB float32 image in [0, 255]. Output: CHW float32 tensor.
    """
    img = cv2.resize(image_f32, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
    img = np.clip(img, 0, 255) / 255.0
    img = (img - IMAGENET_MEAN) / IMAGENET_STD
    return torch.from_numpy(img.transpose(2, 0, 1).copy()).float()

## 4. Dataset

In [ ]:
class SegmentationDataset(Dataset):
    """Image/mask dataset for SegFormer.

    Masks are binarized at 127 (plume = 1, background = 0), matching the label
    convention used by every other model in the study. Training samples are
    augmented with the shared policy of the study.
    """

    def __init__(self, images_dir, masks_dir, augment=False):
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir)
        self.augmentor = Augmentor() if augment else None

        if not self.images_dir.exists():
            raise ValueError(f"Images directory not found: {self.images_dir}")
        if not self.masks_dir.exists():
            raise ValueError(f"Masks directory not found: {self.masks_dir}")

        image_extensions = ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.tiff"]
        image_files = []
        for ext in image_extensions:
            image_files.extend(self.images_dir.glob(ext))
            image_files.extend(self.images_dir.glob(ext.upper()))
        image_files = sorted(set(image_files))

        # Pair every image with its mask by base name.
        self.valid_pairs = []
        missing = []
        for img_file in image_files:
            mask_found = None
            for mask_ext in [".png", ".jpg", ".jpeg", ".bmp"]:
                mask_file = self.masks_dir / (img_file.stem + mask_ext)
                if mask_file.exists():
                    mask_found = mask_file
                    break
            if mask_found:
                self.valid_pairs.append((img_file, mask_found))
            else:
                missing.append(img_file.name)

        if missing:
            print(f"Warning: no masks found for {len(missing)} images "
                  f"(first: {missing[:5]})")

        print(f"Dataset: {len(self.valid_pairs)} image-mask pairs")
        if len(self.valid_pairs) == 0:
            raise ValueError("No image-mask pairs found; check the file names.")

    def __len__(self):
        return len(self.valid_pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.valid_pairs[idx]

        try:
            image = cv2.imread(str(img_path))
            if image is None:
                raise ValueError(f"Could not load image: {img_path}")
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            if mask is None:
                raise ValueError(f"Could not load mask: {mask_path}")

            # Binarize: plume pixels (> 127) -> class 1, background -> class 0.
            mask = (mask > 127).astype(np.uint8)

            # Shared augmentation policy (training split only).
            if self.augmentor is not None:
                image, mask = self.augmentor.augment(image.astype(np.float32), mask)
            else:
                image = image.astype(np.float32)

            pixel_values = preprocess_image(image)
            mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE),
                              interpolation=cv2.INTER_NEAREST)
            labels = torch.as_tensor(mask, dtype=torch.long)

            return {"pixel_values": pixel_values, "labels": labels}

        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            return {
                "pixel_values": torch.zeros((3, IMG_SIZE, IMG_SIZE), dtype=torch.float32),
                "labels": torch.zeros((IMG_SIZE, IMG_SIZE), dtype=torch.long),
            }

## 5. Encoder freezing

In [ ]:
def freeze_encoder(model, freeze_ratio=FREEZE_RATIO):
    """Freeze the first freeze_ratio fraction of encoder parameter tensors.

    SegFormer stores the hierarchical MiT encoder in model.segformer.encoder;
    parameters are frozen in registration order, so the earliest stages are
    frozen first.
    """
    encoder = model.segformer.encoder
    encoder_params = list(encoder.parameters())

    num_to_freeze = int(len(encoder_params) * freeze_ratio)
    for i, param in enumerate(encoder_params):
        param.requires_grad = i >= num_to_freeze

    trainable_encoder = sum(p.numel() for p in encoder.parameters() if p.requires_grad)
    frozen_encoder = sum(p.numel() for p in encoder.parameters() if not p.requires_grad)
    trainable_head = sum(p.numel() for p in model.decode_head.parameters() if p.requires_grad)

    print(f"Frozen encoder parameter tensors: {num_to_freeze}/{len(encoder_params)} "
          f"({freeze_ratio * 100:.0f}%)")
    print(f"Trainable encoder parameters: {trainable_encoder:,}")
    print(f"Frozen encoder parameters:    {frozen_encoder:,}")
    print(f"Trainable decode-head parameters: {trainable_head:,}")

## 6. Training

In [ ]:
def dice_from_logits(logits, labels, smooth=1e-6):
    """Batch Dice of the argmax prediction for the plume class (class 1)."""
    logits_up = F.interpolate(logits, size=labels.shape[-2:],
                              mode="bilinear", align_corners=False)
    preds = torch.argmax(logits_up, dim=1)
    pred_f = (preds == 1).float()
    target_f = (labels == 1).float()
    intersection = (pred_f * target_f).sum()
    union = pred_f.sum() + target_f.sum()
    return ((2.0 * intersection + smooth) / (union + smooth)).item()


def train_segformer():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name()}")
        optimize_memory()

    train_dataset = SegmentationDataset(
        images_dir=f"{TRAIN_DIR}/images",
        masks_dir=f"{TRAIN_DIR}/Masks",
        augment=True,
    )
    val_dataset = SegmentationDataset(
        images_dir=f"{VAL_DIR}/images",
        masks_dir=f"{VAL_DIR}/Masks",
        augment=False,
    )

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)

    print(f"Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)}")

    model = SegformerForSemanticSegmentation.from_pretrained(
        PRETRAINED,
        num_labels=NUM_CLASSES,
        ignore_mismatched_sizes=True,
    )

    freeze_encoder(model, FREEZE_RATIO)
    model = model.to(device)

    # Two parameter groups: slower LR for the (partially frozen) encoder.
    encoder_params = [p for p in model.segformer.encoder.parameters() if p.requires_grad]
    head_params = [p for p in model.decode_head.parameters() if p.requires_grad]

    optimizer = torch.optim.AdamW([
        {"params": encoder_params, "lr": LEARNING_RATE * ENCODER_LR_MULT},
        {"params": head_params, "lr": LEARNING_RATE},
    ], weight_decay=WEIGHT_DECAY)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    print(f"LR encoder: {LEARNING_RATE * ENCODER_LR_MULT:.2e} | LR head: {LEARNING_RATE:.2e}")
    print(f"Epochs: {EPOCHS} | Batch size: {BATCH_SIZE} | Early stopping patience: {PATIENCE}")

    best_val_loss = float("inf")
    selected_val_dice = 0.0
    history = {"train_loss": [], "val_loss": [], "val_dice": [], "lr_head": []}
    patience_counter = 0

    for epoch in range(EPOCHS):
        print(f"\nEpoch {epoch + 1}/{EPOCHS}")
        optimize_memory()

        # ---- Training ----
        model.train()
        train_loss, train_steps = 0.0, 0

        for batch in tqdm(train_loader, desc="Train", leave=False):
            try:
                optimizer.zero_grad()

                pixel_values = batch["pixel_values"].to(device, non_blocking=True)
                labels = batch["labels"].to(device, non_blocking=True)

                outputs = model(pixel_values=pixel_values, labels=labels)
                loss = outputs.loss

                if torch.isnan(loss) or torch.isinf(loss):
                    continue

                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
                optimizer.step()

                train_loss += loss.item()
                train_steps += 1

            except RuntimeError as e:
                if "out of memory" in str(e):
                    print("GPU out of memory in a batch; clearing memory")
                    optimize_memory()
                    continue
                print(f"Runtime error: {e}")
                continue

        if train_steps == 0:
            print("No successful training batches; stopping.")
            break

        avg_train_loss = train_loss / train_steps

        # ---- Validation ----
        model.eval()
        val_loss, val_dice, val_steps = 0.0, 0.0, 0

        with torch.no_grad():
            for batch in tqdm(val_loader, desc="Val", leave=False):
                try:
                    pixel_values = batch["pixel_values"].to(device, non_blocking=True)
                    labels = batch["labels"].to(device, non_blocking=True)

                    outputs = model(pixel_values=pixel_values, labels=labels)
                    loss = outputs.loss

                    if torch.isnan(loss) or torch.isinf(loss):
                        continue

                    val_loss += loss.item()
                    val_dice += dice_from_logits(outputs.logits, labels)
                    val_steps += 1

                except Exception as e:
                    print(f"Validation error: {e}")
                    continue

        if val_steps == 0:
            print("No successful validation batches; stopping.")
            break

        avg_val_loss = val_loss / val_steps
        avg_val_dice = val_dice / val_steps

        scheduler.step()

        history["train_loss"].append(avg_train_loss)
        history["val_loss"].append(avg_val_loss)
        history["val_dice"].append(avg_val_dice)
        history["lr_head"].append(optimizer.param_groups[1]["lr"])

        print(f"Train loss: {avg_train_loss:.4f} | Val loss: {avg_val_loss:.4f} | "
              f"Val Dice: {avg_val_dice:.4f} | LR head: {optimizer.param_groups[1]['lr']:.2e}")

        # Checkpoint selection and early stopping on validation loss.
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            selected_val_dice = avg_val_dice
            patience_counter = 0
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "train_loss": avg_train_loss,
                "val_loss": avg_val_loss,
                "val_dice": avg_val_dice,
                "num_classes": NUM_CLASSES,
                "best_val_loss": best_val_loss,
                "config": {"IMG_SIZE": IMG_SIZE, "PRETRAINED": PRETRAINED},
            }, SAVE_PATH)
            print(f"Saved best model (val loss {best_val_loss:.4f})")
        else:
            patience_counter += 1
            print(f"No improvement: {patience_counter}/{PATIENCE}")

        if patience_counter >= PATIENCE:
            print("Early stopping")
            break

    optimize_memory()
    print(f"\nTraining finished. Best val loss: {best_val_loss:.4f}")
    print(f"Validation Dice of the selected checkpoint: {selected_val_dice:.4f}")
    print(f"Best checkpoint: {SAVE_PATH}")
    return history


history = train_segformer()

# Training curves.
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history["train_loss"], label="Train Loss")
plt.plot(history["val_loss"], label="Val Loss")
plt.title("Loss")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history["val_dice"], label="Val Dice", color="green")
plt.title("Validation Dice")
plt.xlabel("Epoch")
plt.legend()

plt.tight_layout()
plt.show()

## 7. Test-set evaluation (canonical metrics)

The best checkpoint is reloaded; the probability mask is the softmax
probability of class 1 (plume) upsampled to the working resolution, and the
same metric set as for the other models is computed at the native
ground-truth resolution.

In [ ]:
from PIL import Image
from scipy.ndimage import distance_transform_edt


def compute_dice(mask_true, mask_pred, eps=1e-7):
    """Dice coefficient between two binary masks (numpy)."""
    mask_true = mask_true.astype(np.float32)
    mask_pred = mask_pred.astype(np.float32)

    intersection = np.sum(mask_true * mask_pred)
    total = np.sum(mask_true) + np.sum(mask_pred)

    if total == 0:
        # Both masks empty: perfect agreement.
        return 1.0

    return (2.0 * intersection) / (total + eps)


def compute_metrics(mask_true, mask_pred, eps=1e-7):
    """Pixel-wise Accuracy, Precision and Recall for binary segmentation."""
    y_true = mask_true.flatten().astype(np.uint8)
    y_pred = mask_pred.flatten().astype(np.uint8)

    TP = np.sum((y_true == 1) & (y_pred == 1))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    TN = np.sum((y_true == 0) & (y_pred == 0))

    total = TP + TN + FP + FN
    accuracy = (TP + TN) / (total + eps)
    precision = 0.0 if (TP + FP) == 0 else TP / (TP + FP)
    recall = 0.0 if (TP + FN) == 0 else TP / (TP + FN)

    return accuracy, precision, recall


def compute_hausdorff_distances(mask_true, mask_pred):
    """HD95, Average HD and Max HD between two binary masks.

    Distances are computed symmetrically with Euclidean distance transforms
    over the full masks of an image.
    Returns (hd95, avg_hd, max_hd) in pixels.
    """
    true_coords = np.argwhere(mask_true > 0)
    pred_coords = np.argwhere(mask_pred > 0)

    if len(true_coords) == 0 or len(pred_coords) == 0:
        if len(true_coords) == 0 and len(pred_coords) == 0:
            return 0.0, 0.0, 0.0
        # One mask is empty: report the image diagonal as the distance.
        max_dist = np.sqrt(mask_true.shape[0] ** 2 + mask_true.shape[1] ** 2)
        return max_dist, max_dist, max_dist

    distances_pred_to_true = distance_transform_edt(1 - mask_true)
    distances_pred_to_true = distances_pred_to_true[pred_coords[:, 0], pred_coords[:, 1]]

    distances_true_to_pred = distance_transform_edt(1 - mask_pred)
    distances_true_to_pred = distances_true_to_pred[true_coords[:, 0], true_coords[:, 1]]

    all_distances = np.concatenate([distances_pred_to_true, distances_true_to_pred])

    hd95 = np.percentile(all_distances, 95)
    avg_hd = np.mean(all_distances)
    max_hd = np.max(all_distances)

    return hd95, avg_hd, max_hd


def binarize_mask(mask, threshold=0.5):
    """Binarize a mask with the given threshold."""
    return (mask > threshold).astype(np.uint8)


def find_corresponding_mask(img_name, mask_files):
    """Find the ground-truth mask file matching an image file name."""
    img_stem = os.path.splitext(img_name)[0]

    possible_names = [
        img_name,
        f"{img_stem}.png",
        f"{img_stem}.jpg",
        f"{img_stem}.jpeg",
        f"{img_stem}_mask.png",
        f"mask_{img_stem}.png",
        f"{img_stem}_gt.png",
        f"{img_stem}_groundtruth.png",
    ]

    for possible_name in possible_names:
        if possible_name in mask_files:
            return possible_name
    return None


def resize_to_match(mask, target_size):
    """Resize a binary mask to target_size with nearest-neighbour resampling."""
    if isinstance(mask, np.ndarray):
        mask_pil = Image.fromarray((mask * 255).astype(np.uint8))
    else:
        mask_pil = mask

    resized = mask_pil.resize(target_size, resample=Image.NEAREST)
    return (np.array(resized) > 127).astype(np.uint8)

In [ ]:
# Load the best checkpoint.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

eval_model = SegformerForSemanticSegmentation.from_pretrained(
    PRETRAINED, num_labels=NUM_CLASSES, ignore_mismatched_sizes=True,
)
checkpoint = torch.load(SAVE_PATH, map_location="cpu")
eval_model.load_state_dict(checkpoint["model_state_dict"])
eval_model.to(device)
eval_model.eval()
print(f"Loaded checkpoint from epoch {checkpoint.get('epoch', '?')} "
      f"(val loss {checkpoint.get('best_val_loss', float('nan')):.4f})")

@torch.no_grad()
def predict_probability(image_path):
    """Softmax probability of the plume class in [0, 1] at IMG_SIZE resolution."""
    image = cv2.imread(image_path)
    if image is None:
        raise ValueError(f"Could not load image: {image_path}")
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    pixel_values = preprocess_image(image.astype(np.float32)).unsqueeze(0).to(device)

    outputs = eval_model(pixel_values=pixel_values)
    logits = F.interpolate(outputs.logits, size=(IMG_SIZE, IMG_SIZE),
                           mode="bilinear", align_corners=False)
    prob = torch.softmax(logits, dim=1)[0, 1, :, :].cpu().numpy()
    return prob


img_names = sorted([f for f in os.listdir(TEST_IMG_DIR)
                    if f.lower().endswith((".png", ".jpg", ".jpeg"))])
mask_files = sorted([f for f in os.listdir(TEST_MASK_DIR)
                     if f.lower().endswith((".png", ".jpg", ".jpeg"))])

dice_scores, accuracy_scores = [], []
precision_scores, recall_scores = [], []
hd95_scores, avg_hd_scores, max_hd_scores = [], [], []

for i, img_name in enumerate(img_names):
    mask_name = find_corresponding_mask(img_name, mask_files)
    if mask_name is None:
        print(f"Mask not found for {img_name}, skipping")
        continue

    true_mask_img = Image.open(os.path.join(TEST_MASK_DIR, mask_name)).convert("L")
    true_mask_bin = (np.array(true_mask_img) > 127).astype(np.uint8)

    prob = predict_probability(os.path.join(TEST_IMG_DIR, img_name))
    pred_bin = binarize_mask(prob, threshold=PRED_THRESHOLD)
    pred_bin = resize_to_match(pred_bin, true_mask_img.size)

    if true_mask_bin.shape != pred_bin.shape:
        min_h = min(true_mask_bin.shape[0], pred_bin.shape[0])
        min_w = min(true_mask_bin.shape[1], pred_bin.shape[1])
        true_mask_bin = true_mask_bin[:min_h, :min_w]
        pred_bin = pred_bin[:min_h, :min_w]

    dice_scores.append(compute_dice(true_mask_bin, pred_bin))
    acc, prec, rec = compute_metrics(true_mask_bin, pred_bin)
    accuracy_scores.append(acc)
    precision_scores.append(prec)
    recall_scores.append(rec)
    hd95, avg_hd, max_hd = compute_hausdorff_distances(true_mask_bin, pred_bin)
    hd95_scores.append(hd95)
    avg_hd_scores.append(avg_hd)
    max_hd_scores.append(max_hd)

    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1}/{len(img_names)}")

print("\n" + "=" * 70)
print("TEST-SET RESULTS: SegFormer-B3")
print("=" * 70)
print(f"Mean Dice Coefficient: {np.mean(dice_scores):.4f} +/- {np.std(dice_scores):.4f}")
print(f"Mean Accuracy:         {np.mean(accuracy_scores):.4f} +/- {np.std(accuracy_scores):.4f}")
print(f"Mean Precision:        {np.mean(precision_scores):.4f} +/- {np.std(precision_scores):.4f}")
print(f"Mean Recall:           {np.mean(recall_scores):.4f} +/- {np.std(recall_scores):.4f}")
print(f"Mean HD95:             {np.mean(hd95_scores):.4f} +/- {np.std(hd95_scores):.4f}")
print(f"Mean Average HD:       {np.mean(avg_hd_scores):.4f} +/- {np.std(avg_hd_scores):.4f}")
print(f"Mean Max HD:           {np.mean(max_hd_scores):.4f} +/- {np.std(max_hd_scores):.4f}")

## 8. Export predictions for the ensemble

In [ ]:
os.makedirs(PROBABILITY_EXPORT_DIR, exist_ok=True)
os.makedirs(BINARY_EXPORT_DIR, exist_ok=True)

for img_name in img_names:
    img_stem = os.path.splitext(img_name)[0]

    prob = predict_probability(os.path.join(TEST_IMG_DIR, img_name))

    prob_png = (prob * 255).astype(np.uint8)
    cv2.imwrite(os.path.join(PROBABILITY_EXPORT_DIR, f"{img_stem}_pred.png"), prob_png)

    binary_png = ((prob > PRED_THRESHOLD).astype(np.uint8) * 255)
    cv2.imwrite(os.path.join(BINARY_EXPORT_DIR, f"{img_stem}_pred.png"), binary_png)

print(f"Probability masks written to: {PROBABILITY_EXPORT_DIR}")
print(f"Binary masks written to:      {BINARY_EXPORT_DIR}")